In [1]:
using CairoMakie, LaTeXStrings, JLD2, FFTW, DSP, LsqFit, Statistics

## data = load_object("../Ising RA Data/N_22_entropy_data/IsingRA_Svn_N=22_J=1.0_lambda=6.1_h=1.05_g=0.45.jld2")
## data[:times, :entropy_A, :entropy_half, :purity_A, :purity_half, :overlap_overall, :overlap_A, :overlap_half];

function cumulative_avg(t_vals, f_vals)
    avg, integral = zeros(length(f_vals)), 0.0
    avg[1] = f_vals[1]
    for i in 2:length(t_vals)
        integral += (f_vals[i] + f_vals[i-1]) / 2 * (t_vals[i] - t_vals[i-1])
        avg[i] = integral / t_vals[i]
    end
    return avg
end

@inline function safe_decay_term(x, k)
    kx = k * x
    # If kx is small, use Taylor expansion 
    if abs(kx) < 1e-7
        return 1.0 - (kx / 2.0) + (kx^2 / 6.0)
    else
        return (1.0 - exp(-kx)) / kx
    end
end;

@views function cumulative_entropy_model(x, p)
    A, k = p[1], p[2]
    return @. A * (1.0 - safe_decay_term(x, k))
end;


## Entropy and Mutual Information

In [ ]:
N = 18;

fig = Figure(size = (2*800, 3*600))
xmin = 0; xmax = nothing; 
ymin = nothing; ymax = nothing;

axSA = Axis(fig[1,1], 
    xlabel = L"\lambda t", xlabelsize = 24, 
    ylabel = L"S_A(t) ", ylabelsize = 24,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (0, 50, ymin, 1.1)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

axSA_bar = Axis(fig[1,2], 
    xlabel = L"\lambda t", xlabelsize = 24, xscale = log10, 
    ylabel = L"\bar{S}_A(t)", ylabelsize = 24, yscale = log10,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (1., 1e3, 0.1, 1.1)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

axS_half = Axis(fig[2,1], 
    xlabel = L"\lambda t", xlabelsize = 24, 
    ylabel = L"S_{N/2}(t)", ylabelsize = 24,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (0, 50, ymin, N/2+1)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

axS_half_bar = Axis(fig[2,2], 
    xlabel = L"\lambda t", xlabelsize = 24, xscale = log10, 
    ylabel = L"\bar{S}_{N/2}(t)", ylabelsize = 24, yscale = log10,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (1., 10^(1.6), 0.3, N/2+1)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

axI = Axis(fig[3,1], 
    xlabel = L"\lambda t", xlabelsize = 24, 
    ylabel = L"I(t)", ylabelsize = 24,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (0, 50, ymin, N)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

axIbar = Axis(fig[3,2], 
    xlabel = L"\lambda t", xlabelsize = 24, xscale = log10, 
    ylabel = L"\bar{I}(t)", ylabelsize = 24, yscale = log10,
    xgridvisible = true, xminorticks = IntervalsBetween(10), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorticks = IntervalsBetween(10), yminorticksvisible = true, yminorgridvisible = true,
    xticklabelsize = 18, 
    yticklabelsize = 18, 
    limits = (1., 10^(1.6), 0.3, N)
)
vlines!([0], color = :black, linewidth = 1); hlines!([0], color = :black, linewidth = 1);

###################################################################################################################
###################################################################################################################

l_arr = 10 .^(-1.:0.05:3) / N;
lpts = length(l_arr)
colors = [(Makie.resample_cmap(:plasma, lpts))...]

k_arr = zeros(lpts);
A_arr = zeros(lpts);
r2_arr = zeros(lpts);

p0 = [N, l_arr[1]]

for (i,l) in enumerate(l_arr)
    # if i%4≠0
    #    continue
    #end

    ## Load Data
    file_path = "../Ising RA Data/adaptive_lambda_sweep_6.8.26/adaptive_N$(N)/adaptive_IsingRA_Svn_N=$(N)_J=1.0_lambda=$(l)_h=1.05_g=0.0.jld2"
    if !isfile(file_path)
        continue 
    end
    
    data = load_object(file_path)
    tarr      = data[:times];
    SA_arr    = data[:entropy_A] ./ log(2);
    S_half_arr = data[:entropy_half] ./ log(2);
    I_arr = ( 2 .* S_half_arr .- SA_arr );

    ## Averages 
    SA_bar_arr = cat([0], cumulative_avg(tarr[2:end], SA_arr[2:end]), dims=1);
    S_half_bar_arr = cat([0], cumulative_avg(tarr[2:end], S_half_arr[2:end]), dims=1);
    Ibar_arr = cat([0], cumulative_avg(tarr[2:end], I_arr[2:end]), dims=1);

    lines!(axSA, l .* tarr, SA_arr, color = colors[i], linewidth = 2)
    lines!(axSA_bar,  l .* tarr, SA_bar_arr, color = colors[i], linewidth = 2)
    
    lines!(axS_half,  l .* tarr, S_half_arr, color = colors[i], linewidth = 2)
    lines!(axS_half_bar,  l .* tarr, S_half_bar_arr, color = colors[i], linewidth = 2)

    lines!(axI,  l .* tarr, I_arr, color = colors[i], linewidth = 2)
    lines!(axIbar,  l .* tarr, Ibar_arr, color = colors[i], linewidth = 2)

    ## fitting
    fit_Ibar = curve_fit(cumulative_entropy_model, tarr[2:end], Ibar_arr[2:end], p0, lower = [0.,0.], upper = [Inf, Inf])
    A_arr[i], k_arr[i],  = coef(fit_Ibar);
    r2_arr[i] = 1 - ( sum(fit_Ibar.resid .^ 2) / sum((Ibar_arr .- mean(Ibar_arr)) .^ 2) );

    p0 = fit_Ibar.param;

end

Colorbar(fig[1, 3], colormap = :plasma, limits = (minimum(N .* l_arr), maximum(N .* l_arr)), scale = log10, ticklabelsize = 24, label = L"\lambda N", labelsize = 24)

hlines!(axSA, [1], linestyle = :dash, color = [:green], linewidth = 2)
hlines!(axSA_bar, [1], linestyle = :dash, color = [:green], linewidth = 2)
hlines!(axS_half, [1, N/2*(2-1/log(2)), N÷2], linestyle = :dash, color = [:green, :blue, :cyan], linewidth = 2)
hlines!(axS_half_bar, [1, N/2*(2-1/log(2)), N÷2], linestyle = :dash, color = [:green, :blue, :cyan], linewidth = 2)
hlines!(axI, [1, N*(2-1/log(2))-1, N-1], linestyle = :dash, color = [:green, :blue, :cyan], linewidth = 2)
hlines!(axIbar, [1, N*(2-1/log(2))-1, N-1], linestyle = :dash, color = [:green, :blue, :cyan], linewidth = 2)


display(fig)

In [ ]:
fit_fig = Figure(size = (2*800, 600))

axk_fit = Axis(fit_fig[1,1], 
    xlabel = L"\lambda N", xlabelsize = 24, xscale = log10,
    ylabel = L"\tau / N", ylabelsize = 24, 
    xgridvisible = true, xminorticks = IntervalsBetween(10), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorgridvisible = true, yminorticks = IntervalsBetween(10), yminorticksvisible = true, 
    xticklabelsize = 18, 
    yticklabelsize = 18, 
)

axA_fit = Axis(fit_fig[1,2], 
    xlabel = L"\lambda N", xlabelsize = 24, xscale = log10,
    ylabel = L"S_\infty / N", ylabelsize = 24, 
    xgridvisible = true, xminorticks = IntervalsBetween(10), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorgridvisible = true, yminorticks = IntervalsBetween(10), yminorticksvisible = true, 
    xticklabelsize = 18, 
    yticklabelsize = 18, 
)

N_arr = 12:2:20; Npts = length(N_arr)

colors = [(Makie.resample_cmap(:viridis, Npts))...]

alphaL_arr = zeros(Npts);
AL_arr = zeros(Npts);
CL_arr = zeros(Npts);

alphaR_arr = zeros(Npts);
AR_arr = zeros(Npts)
CR_arr = zeros(Npts);

for (n_idx, N) in enumerate(N_arr)
    l_arr = 10 .^(-1.:0.05:3.) / N;
    lpts = length(l_arr)
    
    k_arr = zeros(lpts)
    A_arr = zeros(lpts)
    
    p0 = [Float64(N), l_arr[1]] 

    for (i, l) in enumerate(l_arr)
        
        file_path = "../Ising RA Data/adaptive_lambda_sweep_6.8.26/adaptive_N$(N)/adaptive_IsingRA_Svn_N=$(N)_J=1.0_lambda=$(l)_h=1.05_g=0.0.jld2"
        if !isfile(file_path)
            A_arr[i] = NaN
            k_arr[i] = Inf
            continue 
        end
        
        data = load_object(file_path);
        tarr       = data[:times]
        SA_arr     = data[:entropy_A] ./ log(2)
        S_half_arr = data[:entropy_half] ./ log(2)
        I_arr      = ( 2 .* S_half_arr .- SA_arr )

        # Averages 
        Ibar_arr = cat([0], cumulative_avg(tarr[2:end], I_arr[2:end]), dims=1)

        # Fitting
        fit_x = l .* tarr[1. .< l .* tarr .< 10 ^ 1.5]
        fit_y = Ibar_arr[1. .< l .* tarr .< 10 ^ 1.5]
        
        fit_Ibar = curve_fit(cumulative_entropy_model, fit_x, fit_y, p0, lower = [0.0, 0.0], upper = [Inf, Inf])
        A_arr[i] = fit_Ibar.param[1]
        k_arr[i] = fit_Ibar.param[2]

        p0 = fit_Ibar.param 
    end
    
    lines!(axk_fit, l_arr .* N, (1 ./ k_arr) ./ N, color = colors[n_idx], linewidth = 2, label = "N = $N")
    lines!(axA_fit, l_arr .* N, A_arr ./ N, color = colors[n_idx], linewidth = 2, label = "N = $N")

    idx = argmin((k_arr[2:end-1]))
    l0 = l_arr[idx] .* N; 
    τ0 = 1 ./k_arr[1] ./ N;
    
    l_L = l_arr[1:idx] .* N;
    τ_L = 1 ./ k_arr[1:idx] ./ N;

    @views abs_power_model_C(x, p) = @. p[2] ./ (abs(x-(l_arr[idx+1] .* N))).^ p[1] + p[3]
        
    fit_L = curve_fit(abs_power_model_C, l_L, τ_L, [2., 1., τ0], lower = [0., 0., 0.], upper=[Inf, Inf, Inf])
    alphaL_arr[n_idx], AL_arr[n_idx], CL_arr[n_idx] = fit_L.param;
    #err_L = margin_error(fit_L, 0.05) 
    println("α : $(round(alphaL_arr[n_idx], sigdigits=3)) , A : $(round(AL_arr[n_idx], sigdigits=3)), C : $(round(CL_arr[n_idx], sigdigits=3)) ")
    println()

    l_R = l_arr[idx:end] .* N;
    τ_R = 1 ./k_arr[idx:end] ./ N;

    @views abs_power_model_CR(x, p) = @. p[2] ./ (abs(x-(l_arr[idx-1] .* N))).^ p[1] + p[3]

    fit_R = curve_fit(abs_power_model_CR, l_R, τ_R, [2., 1., 0.], lower = [0., 0., 0.], upper=[Inf, Inf, Inf])
    alphaR_arr[n_idx], AR_arr[n_idx], CR_arr[n_idx] = fit_R.param;
    #err_R = margin_error(fit_R, 0.05) 
    println("α : $(round(alphaR_arr[n_idx], sigdigits=3)) , A : $(round(AR_arr[n_idx], sigdigits=3)), C : $(round(CR_arr[n_idx], sigdigits=3)) ")
    println("----------------------------")

    if n_idx==Npts
        vlines!(axk_fit, [l0], color = :red, linewidth = 2, linestyle = :dash)
        lines!(axk_fit, l_L, abs_power_model_C(l_L, fit_L.param), color = :red, linewidth = 3, linestyle = :dash)
        lines!(axk_fit, l_R, abs_power_model_CR(l_R, fit_R.param), color = :red, linewidth = 3, linestyle = :dash)
    end
end

Legend(fit_fig[1, 3], axk_fit, "System Size", labelsize = 18, titlesize = 20, framevisible = false)

display(fit_fig)

In [ ]:
fig = Figure(size = (800, 600))
ax_alphaL = Axis(fig[1,1], 
    xlabel = L" N", xlabelsize = 24, #xscale = log10,
    ylabel = L"\alpha^-(N)", ylabelsize = 24, yticklabelcolor = :blue, ylabelcolor = :blue,
    xgridvisible = true, xminorticks = IntervalsBetween(5), xminorticksvisible = true, xminorgridvisible = true,
    ygridvisible = true, yminorgridvisible = true, yminorticks = IntervalsBetween(5), yminorticksvisible = true, 
    xticklabelsize = 18, xticks = 6:2:20,
    yticklabelsize = 18, #yticks = [-0.25,0,0.25], 
    limits = (nothing, nothing, 0., 2.)
)
ax_alphaR = Axis(fig[1,1], 
    ylabel = L"\alpha^+(N)", ylabelsize = 24, yticklabelcolor = :red, yaxisposition = :right, ylabelcolor = :red,
    ygridvisible = false, yminorgridvisible = false, yminorticks = IntervalsBetween(5), yminorticksvisible = true, 
    yticklabelsize = 18, #yticks = [-0.25,0,0.25], 
    limits = (nothing, nothing, 0., 2.)
)
hidespines!(ax_alphaR)
hidexdecorations!(ax_alphaR)

lines!(ax_alphaL, N_arr, alphaL_arr, color = :blue, linewidth = 4)
lines!(ax_alphaR, N_arr, alphaR_arr, color = :red, linewidth = 4)

display(fig)